# 00 - Train/Test split (stratified by amount bins and win/loss)

This notebook creates a train/test split stratifying on a combined key:
- `Opportunity Result Bool` (Won/Loss target)
- `Opportunity Amount USD` quantile bin (qcut)

Outputs are saved to `data/intermediate/` as parquet files.

In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [28]:
DATA_PATH = '../../data/intermediate/cleaned_data.parquet'
TARGET_COL = 'Opportunity Result Bool'
AMOUNT_COL = 'Opportunity Amount USD'

df = pd.read_parquet(DATA_PATH).copy()
print('Loaded:', DATA_PATH)
print('shape:', df.shape)
df[[TARGET_COL, AMOUNT_COL]].head()

Loaded: ../../data/intermediate/cleaned_data.parquet
shape: (75819, 35)


,Opportunity Result Bool,Opportunity Amount USD
0,False,232522
1,True,109767
2,False,150000
3,False,250000
4,False,80000


In [29]:
# Keep rows with valid target/amount
work = df.dropna(subset=[TARGET_COL, AMOUNT_COL]).copy()

# Build amount quantile bins (fallback to fewer bins if needed)
candidate_q = [10, 8, 6, 5, 4, 3, 2]
selected_q = None
for q in candidate_q:
    try:
        bins = pd.qcut(work[AMOUNT_COL], q=q, duplicates='drop')
        key = work[TARGET_COL].astype(int).astype(str) + '__' + bins.astype(str)
        if key.value_counts().min() >= 2:
            selected_q = q
            work['amount_qbin'] = bins
            work['stratify_key'] = key
            break
    except ValueError:
        continue

if selected_q is None:
    raise ValueError('Could not build a valid stratification key with min class count >= 2')

print('Rows used for split:', len(work))
print('qcut bins selected:', selected_q)
print('stratify classes:', work['stratify_key'].nunique())
print('min class count:', int(work['stratify_key'].value_counts().min()))
work[['amount_qbin', 'stratify_key']].head()

Rows used for split: 75819
qcut bins selected: 10
stratify classes: 20
min class count: 829


,amount_qbin,stratify_key
0,"(140000.0, 232522.0]","0__(140000.0, 232522.0]"
1,"(100000.0, 140000.0]","1__(100000.0, 140000.0]"
2,"(140000.0, 232522.0]","0__(140000.0, 232522.0]"
3,"(232522.0, 1000000.0]","0__(232522.0, 1000000.0]"
4,"(65789.0, 100000.0]","0__(65789.0, 100000.0]"


In [30]:
TRAIN_SIZE = 0.7
RANDOM_STATE = 42

train_df, test_df = train_test_split(
    work,
    train_size=TRAIN_SIZE,
    random_state=RANDOM_STATE,
    stratify=work['stratify_key']
)

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('train ratio:', round(len(train_df) / len(work), 4))
print('test ratio:', round(len(test_df) / len(work), 4))

train shape: (53073, 37)
test shape: (22746, 37)
train ratio: 0.7
test ratio: 0.3


In [31]:
def summarize_split(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    out = (
        frame.groupby('stratify_key', observed=False)
        .size()
        .rename('n')
        .to_frame()
    )
    out['pct'] = out['n'] / out['n'].sum()
    out['split'] = name
    return out.reset_index()

sum_train = summarize_split(train_df, 'train')
sum_test = summarize_split(test_df, 'test')
sum_all = summarize_split(work, 'all')

summary = pd.concat([sum_all, sum_train, sum_test], ignore_index=True)
display(summary.head(15))

pivot = summary.pivot(index='stratify_key', columns='split', values='pct').fillna(0)
pivot['abs_diff_train_vs_all'] = (pivot['train'] - pivot['all']).abs()
pivot['abs_diff_test_vs_all'] = (pivot['test'] - pivot['all']).abs()
print('max abs diff train vs all:', float(pivot['abs_diff_train_vs_all'].max()))
print('max abs diff test  vs all:', float(pivot['abs_diff_test_vs_all'].max()))
display(pivot.sort_values('abs_diff_test_vs_all', ascending=False).head(15))

,stratify_key,n,pct,split
0,"0__(0.999, 6500.0]",4293,0.056622,all
1,"0__(100000.0, 140000.0]",5513,0.072713,all
2,"0__(13000.0, 20000.0]",6325,0.083422,all
3,"0__(140000.0, 232522.0]",6649,0.087696,all
4,"0__(20000.0, 30000.0]",5483,0.072317,all
5,"0__(232522.0, 1000000.0]",6040,0.079663,all
6,"0__(30000.0, 50000.0]",8839,0.116580,all
7,"0__(50000.0, 65789.0]",2841,0.037471,all
8,"0__(6500.0, 13000.0]",5128,0.067635,all
9,"0__(65789.0, 100000.0]",7477,0.098616,all


max abs diff train vs all: 1.145518559660108e-05
max abs diff test  vs all: 2.6728262778878686e-05


split,all,test,train,abs_diff_train_vs_all,abs_diff_test_vs_all
stratify_key,,,,,
"1__(13000.0, 20000.0]",0.026537,0.026510,0.026548,0.000011,0.000027
"0__(13000.0, 20000.0]",0.083422,0.083399,0.083432,0.000010,0.000023
"1__(232522.0, 1000000.0]",0.018531,0.018509,0.018541,0.000010,0.000022
"0__(6500.0, 13000.0]",0.067635,0.067616,0.067643,0.000008,0.000018
"1__(30000.0, 50000.0]",0.028383,0.028401,0.028376,0.000007,0.000017
"1__(6500.0, 13000.0]",0.032604,0.032621,0.032597,0.000007,0.000017
"1__(0.999, 6500.0]",0.043947,0.043964,0.043939,0.000007,0.000017
"0__(50000.0, 65789.0]",0.037471,0.037457,0.037477,0.000006,0.000014
"1__(100000.0, 140000.0]",0.010934,0.010947,0.010928,0.000006,0.000013


In [32]:
TRAIN_PATH = '../../data/intermediate/df_train_stratified.parquet'
TEST_PATH = '../../data/intermediate/df_test_stratified.parquet'
SUMMARY_PATH = '../../data/intermediate/df_train_test_stratified_summary.csv'

# Keep helper columns in output for traceability (cast interval bins to string for parquet compatibility)
train_out = train_df.copy()
test_out = test_df.copy()
train_out['amount_qbin'] = train_out['amount_qbin'].astype(str)
test_out['amount_qbin'] = test_out['amount_qbin'].astype(str)
train_out.to_parquet(TRAIN_PATH, index=False)
test_out.to_parquet(TEST_PATH, index=False)
summary.to_csv(SUMMARY_PATH, index=False)

print('saved:', TRAIN_PATH)
print('saved:', TEST_PATH)
print('saved:', SUMMARY_PATH)

saved: ../../data/intermediate/df_train_stratified.parquet
saved: ../../data/intermediate/df_test_stratified.parquet
saved: ../../data/intermediate/df_train_test_stratified_summary.csv
